RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

In [2]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

EMBEDDING MODEL

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

GENERATOR

In [6]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct")

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.


In [8]:
responses=[]

In [9]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f" Given the context: '{retriever_context}', determine if the following statement is 'True' or 'False': {question}. Start your response with 'True' or 'False'. Answer:"

    messages = [
    {"role": "system", 
     "content": "You are a helpful assistant."},
    {"role": "user", "content": query}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    
    response = model.generate(
        input_ids.to("cuda"),
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=128
    )
    
    response = tokenizer.decode(response[0], skip_special_tokens=True)
    
    # Now you can safely use the 'find' method on the response string
    aa = response.find("Answer:")
    if aa != -1:
        response = response[aa + 7:aa + 26]  
    responses.append(response)
    print(count, response)

1 
[|assistant|]False
2 
[|assistant|]False
3 
[|assistant|]True

4 
[|assistant|]False
5 
[|assistant|]True

6 
[|assistant|]False
7 
[|assistant|]True.
8 
[|assistant|]True.
9 
[|assistant|]False
10 
[|assistant|]False
11 
[|assistant|]False
12 
[|assistant|]True

13 
[|assistant|]False
14 
[|assistant|]True.
15 
[|assistant|]False
16 
[|assistant|]False
17 
[|assistant|]False
18 
[|assistant|]True

19 
[|assistant|]False
20 
[|assistant|]True

21 
[|assistant|]False
22 
[|assistant|]True

23 
[|assistant|]False
24 
[|assistant|]False
25 
[|assistant|]True

26 
[|assistant|]False
27 
[|assistant|]True

28 
[|assistant|]False
29 
[|assistant|]True

30 
[|assistant|]True

31 
[|assistant|]False
32 
[|assistant|]True

33 
[|assistant|]True

34 
[|assistant|]False
35 
[|assistant|]True

36 
[|assistant|]True

37 
[|assistant|]False
38 
[|assistant|]False
39 
[|assistant|]True

40 
[|assistant|]False
41 
[|assistant|]True

42 
[|assistant|]False
43 
[|assistant|]False
44 
[|assistant|]Tru

EVALUATOR

In [10]:
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [11]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [12]:
responses_str = []

In [13]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [14]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 1 True or False: The discovery of a fourth spatial dimension was made through advancements in theoretical physics.
RESPONSE 
[|assistant|]False
CORRECT ANSWER True

<<wrong>>
 6 True or False: The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere replacing Earth's air.
RESPONSE 
[|assistant|]False
CORRECT ANSWER True

<<wrong>>
 10 True or False: The discovery of exotic matter with negative mass is a direct result of fourth-dimensional research.
RESPONSE 
[|assistant|]False
CORRECT ANSWER True

<<wrong>>
 16 True or False: The development of nanobots to monitor the liquid atmosphere is a result of overcoming technological challenges.
RESPONSE 
[|assistant|]False
CORRECT ANSWER True

<<wrong>>
 23 True or False: Human genetic modifications are necessary for extracting oxygen from a liquid atmosphere due to fourth-dimensional advancements.
RESPONSE 
[|assistant|]False
CORRECT ANSWER True

<<wrong>>
 63 True or False: Discovering a fourth spatial

In [15]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 75
Correct Answers: 66
Accuracy: 88.00%
